In [0]:
%run ../00-common/config

In [0]:
# Për demonstrim do ta mapojmë user-in që po ekzekuton notebook-un te SP

dbutils.widgets.text(
    "demo_state",
    "SP",
    "Demo seller state"
)

demo_state = (
    dbutils.widgets
    .get("demo_state")
    .strip()
    .upper()
)

current_principal = (
    spark.sql("SELECT session_user() AS user_email")
    .first()["user_email"]
)

print("Current user:", current_principal)
print("Demo seller state:", demo_state)

In [0]:
# Hiq RLS e vjetër përpara rebuild

tables_to_reset = [
    f"{catalog_name}.{gold_schema}.dim_seller",
    f"{catalog_name}.{gold_schema}.fact_order_items",
]

for table_name in tables_to_reset:
    try:
        spark.sql(
            f"ALTER TABLE {table_name} DROP ROW FILTER"
        )
        print(f"Removed existing row filter from {table_name}")
    except Exception:
        print(f"No existing row filter on {table_name}")

In [0]:
# Capture baseline para RLS

baseline_sellers = spark.table(
    f"{catalog_name}.{gold_schema}.dim_seller"
).count()

baseline_items = spark.table(
    f"{catalog_name}.{gold_schema}.fact_order_items"
).count()

baseline_states = (
    spark.table(
        f"{catalog_name}.{gold_schema}.dim_seller"
    )
    .select("seller_state")
    .distinct()
    .count()
)

print("BEFORE RLS")
print("Seller rows:", baseline_sellers)
print("Order item rows:", baseline_items)
print("Seller states:", baseline_states)

In [0]:
# Security mapping,,,, krijo mapping table

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{catalog_name}.{control_schema}.seller_state_access
(
    user_email STRING,
    seller_state STRING,
    active BOOLEAN,
    comment STRING
)
USING DELTA
""")

print(
    f"Created/verified "
    f"{catalog_name}.{control_schema}.seller_state_access"
)

In [0]:
# Mapo current user te SP, do përdorim MERGE që rerun-i i notebook-ut të mos krijojë duplicate mapping

assignment_df = spark.createDataFrame(
    [
        (
            current_principal,
            demo_state,
            True,
            "Bonus 3 seller-state RLS demo"
        )
    ],
    [
        "user_email",
        "seller_state",
        "active",
        "comment"
    ]
)

assignment_df.createOrReplaceTempView(
    "current_rls_assignment"
)

spark.sql(f"""
MERGE INTO
    {catalog_name}.{control_schema}.seller_state_access AS target

USING
    current_rls_assignment AS source

ON lower(target.user_email) = lower(source.user_email)

WHEN MATCHED THEN
    UPDATE SET
        target.seller_state = source.seller_state,
        target.active = source.active,
        target.comment = source.comment

WHEN NOT MATCHED THEN
    INSERT (
        user_email,
        seller_state,
        active,
        comment
    )
    VALUES (
        source.user_email,
        source.seller_state,
        source.active,
        source.comment
    )
""")

spark.table(
    f"{catalog_name}.{control_schema}.seller_state_access"
).show(truncate=False)

In [0]:
# Krijo bridge seller → state

spark.sql(f"""
CREATE OR REPLACE TABLE
{catalog_name}.{control_schema}.seller_security_bridge
USING DELTA
AS

SELECT DISTINCT
    seller_id,
    seller_state

FROM
    {catalog_name}.{gold_schema}.dim_seller

WHERE
    seller_id IS NOT NULL
    AND seller_state IS NOT NULL
""")

bridge = spark.table(
    f"{catalog_name}.{control_schema}.seller_security_bridge"
)

print("Security bridge rows:", bridge.count())

bridge.show(5, truncate=False)

In [0]:
# RLS function #1 — Dimension

spark.sql(f"""
CREATE OR REPLACE FUNCTION
{catalog_name}.{gold_schema}.seller_state_filter(state STRING)

RETURNS BOOLEAN

RETURN EXISTS (

    SELECT 1

    FROM
        {catalog_name}.{control_schema}.seller_state_access AS access

    WHERE
        access.active = TRUE

        AND lower(access.user_email)
            = lower(session_user())

        AND (
            access.seller_state = '*'
            OR access.seller_state = state
        )
)
""")

print("Created seller_state_filter()")

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION
{catalog_name}.{gold_schema}.seller_id_filter(input_seller_id STRING)

RETURNS BOOLEAN

RETURN EXISTS (

    SELECT 1

    FROM
        {catalog_name}.{control_schema}.seller_security_bridge AS bridge

    INNER JOIN
        {catalog_name}.{control_schema}.seller_state_access AS access

        ON (
            access.seller_state = '*'
            OR access.seller_state = bridge.seller_state
        )

    WHERE
        bridge.seller_id = input_seller_id

        AND access.active = TRUE

        AND lower(access.user_email)
            = lower(session_user())
)
""")

print("Created seller_id_filter()")

In [0]:
spark.sql(f"""
ALTER TABLE
    {catalog_name}.{gold_schema}.dim_seller

SET ROW FILTER
    {catalog_name}.{gold_schema}.seller_state_filter

ON (seller_state)
""")

print(
    "RLS applied to "
    f"{catalog_name}.{gold_schema}.dim_seller"
)

In [0]:
spark.sql(f"""
ALTER TABLE
    {catalog_name}.{gold_schema}.fact_order_items

SET ROW FILTER
    {catalog_name}.{gold_schema}.seller_id_filter

ON (seller_id)
""")

print(
    "RLS applied to "
    f"{catalog_name}.{gold_schema}.fact_order_items"
)

In [0]:
spark.sql(f"""
SELECT
    table_catalog,
    table_schema,
    table_name,
    filter_name,
    target_columns
FROM
    {catalog_name}.information_schema.row_filters
WHERE
    table_schema = '{gold_schema}'
    AND table_name IN (
        'dim_seller',
        'fact_order_items'
    )
ORDER BY
    table_name
""").show(truncate=False)

In [0]:
print("Current user:", current_principal)
print("Expected state:", demo_state)

spark.sql(f"""
SELECT
    seller_state,
    COUNT(*) AS seller_count

FROM
    {catalog_name}.{gold_schema}.dim_seller

GROUP BY
    seller_state

ORDER BY
    seller_state
""").show()

In [0]:
spark.sql(f"""
SELECT
    bridge.seller_state,
    COUNT(*) AS item_rows

FROM
    {catalog_name}.{gold_schema}.fact_order_items AS items

INNER JOIN
    {catalog_name}.{control_schema}.seller_security_bridge AS bridge

ON
    items.seller_id = bridge.seller_id

GROUP BY
    bridge.seller_state

ORDER BY
    bridge.seller_state
""").show()

In [0]:
secured_sellers = spark.table(
    f"{catalog_name}.{gold_schema}.dim_seller"
).count()

secured_items = spark.table(
    f"{catalog_name}.{gold_schema}.fact_order_items"
).count()

secured_states = (
    spark.table(
        f"{catalog_name}.{gold_schema}.dim_seller"
    )
    .select("seller_state")
    .distinct()
    .count()
)

print("BEFORE RLS")
print("Seller rows:", baseline_sellers)
print("Order item rows:", baseline_items)
print("Seller states:", baseline_states)

print()

print("AFTER RLS")
print("Seller rows:", secured_sellers)
print("Order item rows:", secured_items)
print("Seller states:", secured_states)

In [0]:
if demo_state != "*":

    violations = spark.sql(f"""
    SELECT COUNT(*) AS violations

    FROM
        {catalog_name}.{gold_schema}.fact_order_items AS items

    INNER JOIN
        {catalog_name}.{control_schema}.seller_security_bridge AS bridge

    ON
        items.seller_id = bridge.seller_id

    WHERE
        bridge.seller_state <> '{demo_state}'
    """).first()["violations"]

    print(
        "Rows visible outside assigned state:",
        violations
    )

    assert violations == 0, (
        "RLS validation failed: "
        "user can see another seller state."
    )

    print("✓ Lakehouse RLS validation passed")